# CineRec — Demostración del Sistema de Recomendación

Este notebook muestra en vivo cómo funciona cada modelo de recomendación del sistema CineRec.

Puedes cambiar el `USER_ID` o el `MOVIE_TITLE` para probar con diferentes usuarios y películas.

## 1. Configuración

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
from pathlib import Path

from surprise import Dataset, Reader, SVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 100

# =====================
# PARÁMETROS — modifica aquí
USER_ID    = 1       # ID del usuario para las recomendaciones personalizadas
MOVIE_TITLE = 'Toy Story'  # Película para el recomendador content-based
TOP_N      = 10      # Número de recomendaciones
# =====================

BASE_DIR = Path('..') / 'data' / 'processed'
movies  = pd.read_csv(BASE_DIR / 'movies_clean.csv')
users   = pd.read_csv(BASE_DIR / 'users_clean.csv')
ratings = pd.read_csv(BASE_DIR / 'ratings_clean.csv')

movies['genres_parsed'] = movies['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

print(f'Dataset cargado. Usuario seleccionado: {USER_ID} | Película: {MOVIE_TITLE}')

## 2. Perfil del usuario

In [ ]:
user_info = users[users['userId'] == USER_ID].iloc[0]
user_ratings = ratings[ratings['userId'] == USER_ID]
user_ratings_movies = user_ratings.merge(movies[['movieId','title','genres_parsed']], on='movieId')

age_labels = {1:'<18',18:'18-24',25:'25-34',35:'35-44',45:'45-49',50:'50-55',56:'56+'}

print(f'=== PERFIL DEL USUARIO {USER_ID} ===')
print(f'Género:       {user_info["gender"]}')
print(f'Edad:         {age_labels.get(user_info["age"], user_info["age"])}')
print(f'Total ratings:{len(user_ratings)}')
print(f'Rating medio: {user_ratings["rating"].mean():.2f}')
print()
print('Últimas 5 películas valoradas:')
last5 = user_ratings_movies.sort_values('rating', ascending=False).head(5)
for _, row in last5.iterrows():
    print(f'  ⭐ {row["rating"]} — {row["title"]}')

In [ ]:
# Géneros favoritos del usuario
all_user_genres = [g for genres in user_ratings_movies['genres_parsed'] for g in genres]
genre_counts = pd.Series(all_user_genres).value_counts().head(8)

plt.figure(figsize=(10, 4))
plt.bar(genre_counts.index, genre_counts.values, color='steelblue', edgecolor='white')
plt.title(f'Géneros más vistos por el Usuario {USER_ID}', fontsize=13, fontweight='bold')
plt.ylabel('Número de películas')
plt.tight_layout()
plt.show()

## 3. Recomendador 1 — Popularity

In [ ]:
movie_stats = (
    ratings.groupby('movieId')
    .agg(avg_rating=('rating','mean'), count=('rating','count'))
    .reset_index()
)

watched = ratings[ratings['userId']==USER_ID]['movieId'].tolist()

top_popular = (
    movie_stats[
        (movie_stats['count'] >= 50) &
        (~movie_stats['movieId'].isin(watched))
    ]
    .sort_values('avg_rating', ascending=False)
    .head(TOP_N)
    .merge(movies[['movieId','title','genres_parsed']], on='movieId')
)

print(f'🏆 TOP {TOP_N} PELÍCULAS POPULARES (no vistas por usuario {USER_ID})')
print('─' * 60)
for i, row in enumerate(top_popular.itertuples(), 1):
    genres = ', '.join(row.genres_parsed[:3])
    print(f'{i:2}. {row.title}')
    print(f'    ⭐ {row.avg_rating:.2f} | 🗳️  {row.count:,} votos | 🎭 {genres}')

## 4. Recomendador 2 — Collaborative Filtering (SVD)

In [ ]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId','movieId','rating']], reader)
trainset = data.build_full_trainset()

svd = SVD(n_factors=100, n_epochs=20, random_state=42)
svd.fit(trainset)

candidates = movies[~movies['movieId'].isin(watched)]['movieId'].tolist()
preds = [(mid, svd.predict(USER_ID, mid).est) for mid in candidates]
preds.sort(key=lambda x: x[1], reverse=True)
top_svd = preds[:TOP_N]

print(f'🤝 TOP {TOP_N} RECOMENDACIONES — Collaborative Filtering (Usuario {USER_ID})')
print('─' * 60)
for i, (mid, score) in enumerate(top_svd, 1):
    title = movies[movies['movieId']==mid]['title'].values[0]
    genres = movies[movies['movieId']==mid]['genres_parsed'].values[0]
    print(f'{i:2}. {title}')
    print(f'    ⭐ Predicción: {score:.2f} | 🎭 {", ".join(genres[:3])}')

In [ ]:
# Visualización ratings predichos
svd_titles = [movies[movies['movieId']==mid]['title'].values[0][:30] for mid, _ in top_svd]
svd_scores = [score for _, score in top_svd]

plt.figure(figsize=(12, 5))
bars = plt.barh(svd_titles, svd_scores, color='steelblue', edgecolor='white')
plt.title(f'Ratings Predichos — SVD (Usuario {USER_ID})', fontsize=13, fontweight='bold')
plt.xlabel('Rating predicho')
plt.xlim(0, 5.5)
plt.gca().invert_yaxis()
for bar, val in zip(bars, svd_scores):
    plt.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Recomendador 3 — Content-Based (TF-IDF)

In [ ]:
cb_movies = movies.copy().reset_index(drop=True)
cb_movies['genres_text'] = cb_movies['genres_parsed'].apply(lambda g: ' '.join(g))

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(cb_movies['genres_text'])
sim_matrix = cosine_similarity(tfidf_matrix)

mid_to_idx = {row['movieId']: idx for idx, row in cb_movies.iterrows()}

matches = cb_movies[cb_movies['title'].str.lower() == MOVIE_TITLE.lower()]

if matches.empty:
    print(f'Película "{MOVIE_TITLE}" no encontrada.')
else:
    movie_pos = cb_movies.index.get_loc(matches.index[0])
    source_genres = matches.iloc[0]['genres_parsed']
    scores = list(enumerate(sim_matrix[movie_pos]))
    scores.sort(key=lambda x: x[1], reverse=True)
    top_cb = scores[1:TOP_N+1]

    print(f'🎭 TOP {TOP_N} PELÍCULAS SIMILARES A "{MOVIE_TITLE}"')
    print(f'   Géneros: {", ".join(source_genres)}')
    print('─' * 60)
    for i, (idx, score) in enumerate(top_cb, 1):
        row = cb_movies.iloc[idx]
        print(f'{i:2}. {row["title"]}')
        print(f'    🔗 Similitud: {score:.4f} | 🎭 {", ".join(row["genres_parsed"][:3])}')

## 6. Recomendador 4 — Classification (Random Forest)

In [ ]:
all_genres = sorted(set(g for genres in movies['genres_parsed'] for g in genres))
for genre in all_genres:
    movies[f'genre_{genre}'] = movies['genres_parsed'].apply(lambda g: 1 if genre in g else 0)

le = LabelEncoder()
users = users.copy()
users['gender_encoded'] = le.fit_transform(users['gender'])

genre_cols   = [f'genre_{g}' for g in all_genres]
feature_cols = ['userId','movieId','year','age','gender_encoded','occupation'] + genre_cols

df_all = ratings.merge(movies[['movieId','year']+genre_cols], on='movieId')
df_all = df_all.merge(users[['userId','age','gender_encoded','occupation']], on='userId')
df_all['liked'] = (df_all['rating'] >= 4).astype(int)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(df_all[feature_cols], df_all['liked'])

user = users[users['userId']==USER_ID].iloc[0]
candidates = movies[~movies['movieId'].isin(watched)].copy()
candidates['userId'] = USER_ID
candidates['age'] = user['age']
candidates['gender_encoded'] = user['gender_encoded']
candidates['occupation'] = user['occupation']

probs = rf.predict_proba(candidates[feature_cols])[:, 1]
candidates = candidates.copy()
candidates['prob'] = probs
top_rf = candidates.sort_values('prob', ascending=False).head(TOP_N)

print(f'🤖 TOP {TOP_N} RECOMENDACIONES — Random Forest (Usuario {USER_ID})')
print('─' * 60)
for i, row in enumerate(top_rf.itertuples(), 1):
    genres = ', '.join(row.genres_parsed[:3])
    print(f'{i:2}. {row.title}')
    print(f'    💚 Probabilidad de gustar: {row.prob*100:.1f}% | 🎭 {genres}')

In [ ]:
# Visualización probabilidades
rf_titles = [row.title[:35] for row in top_rf.itertuples()]
rf_probs  = [row.prob * 100 for row in top_rf.itertuples()]
colors = ['#4CAF50' if p >= 70 else '#FFC107' if p >= 50 else '#F44336' for p in rf_probs]

plt.figure(figsize=(12, 5))
bars = plt.barh(rf_titles, rf_probs, color=colors, edgecolor='white')
plt.title(f'Probabilidad de Gustar — Random Forest (Usuario {USER_ID})', fontsize=13, fontweight='bold')
plt.xlabel('Probabilidad (%)')
plt.xlim(0, 115)
plt.gca().invert_yaxis()
for bar, val in zip(bars, rf_probs):
    plt.text(val + 1, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Comparativa visual entre modelos

In [ ]:
# Películas que aparecen en más de un modelo
pop_set  = set(top_popular['movieId'].tolist())
svd_set  = set(mid for mid, _ in top_svd)
cb_set   = set(cb_movies.iloc[idx]['movieId'] for idx, _ in top_cb) if not matches.empty else set()
rf_set   = set(top_rf['movieId'].tolist())

all_sets = {'Popularity': pop_set, 'SVD': svd_set, 'Content-Based': cb_set, 'Random Forest': rf_set}

print('=== SOLAPAMIENTO ENTRE MODELOS ===')
models = list(all_sets.keys())
for i, m1 in enumerate(models):
    for m2 in models[i+1:]:
        overlap = all_sets[m1] & all_sets[m2]
        print(f'{m1} ∩ {m2}: {len(overlap)} películas en común')
        if overlap:
            for mid in list(overlap)[:3]:
                title = movies[movies['movieId']==mid]['title'].values
                if len(title): print(f'   → {title[0]}')